# Lab 5: Xây dựng mô hình RNN cho bài toán Nhận dạng Thực thể có tên (NER)

## Task 1: Tải và Tiền xử lý Dữ liệu

### 1. Tải dữ liệu từ Hugging Face

In [ ]:
! pip install -U datasets==2.19.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 9.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


In [ ]:
import datasets
print(datasets.__version__)

2.19.1


In [ ]:
from datasets import load_dataset

dataset = load_dataset("conll2003")

print(dataset)

### 2. Trích xuất câu và nhãn

In [ ]:
train_sentences = dataset["train"]["tokens"]
train_tags = dataset["train"]["ner_tags"]

label_names = dataset["train"].features["ner_tags"].feature.names

train_tags_str = [[label_names[tag] for tag in seq] for seq in train_tags]

print("Sample sentence:", train_sentences[0])
print("Labels sentence:", train_tags_str[0])

Sample sentence: ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
Labels sentence: ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']


In [ ]:
len(train_sentences)

14041

In [ ]:
val_sentences = dataset["validation"]["tokens"]
val_tags = dataset["validation"]["ner_tags"]

val_label_names = dataset["validation"].features["ner_tags"].feature.names

val_tags_str = [[val_label_names[tag] for tag in seq] for seq in val_tags]

print("Sample sentence:", val_sentences[0])
print("Labels sentence:", val_tags_str[0])


Sample sentence: ['CRICKET', '-', 'LEICESTERSHIRE', 'TAKE', 'OVER', 'AT', 'TOP', 'AFTER', 'INNINGS', 'VICTORY', '.']
Labels sentence: ['O', 'O', 'B-ORG', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [ ]:
len(val_sentences)

3250

In [ ]:
test_sentences = dataset["test"]["tokens"]
test_tags = dataset["test"]["ner_tags"]

test_label_names = dataset["test"].features["ner_tags"].feature.names

test_tags_str = [[test_label_names[tag] for tag in seq] for seq in test_tags]

print("Sample sentence:", test_sentences[0])
print("Labels sentence:", test_tags_str[0])


Sample sentence: ['SOCCER', '-', 'JAPAN', 'GET', 'LUCKY', 'WIN', ',', 'CHINA', 'IN', 'SURPRISE', 'DEFEAT', '.']
Labels sentence: ['O', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'B-PER', 'O', 'O', 'O', 'O']


In [ ]:
len(test_sentences)

3453

### 3. Xây dựng Từ điển (Vocabulary)

In [ ]:
# Xây dựng từ điển word_to_ix
word_to_ix = {"<PAD>": 0, "<UNK>": 1}
for sent in train_sentences:
    for word in sent:
        if word not in word_to_ix:
            word_to_ix[word] = len(word_to_ix)

# Xây dựng từ điển tag_to_ix
unique_tags = sorted(set(tag for seq in train_tags_str for tag in seq))
tag_to_ix = {tag: idx for idx, tag in enumerate(unique_tags)}

print(f"Kích thước từ điển từ (word_to_ix): {len(word_to_ix)}")
print(f"Kích thước từ điển nhãn (tag_to_ix): {len(tag_to_ix)}")

print("\nVí dụ word_to_ix:", list(word_to_ix.items())[:10])
print("Ví dụ tag_to_ix:", tag_to_ix)


Kích thước từ điển từ (word_to_ix): 23625
Kích thước từ điển nhãn (tag_to_ix): 9

Ví dụ word_to_ix: [('<PAD>', 0), ('<UNK>', 1), ('EU', 2), ('rejects', 3), ('German', 4), ('call', 5), ('to', 6), ('boycott', 7), ('British', 8), ('lamb', 9)]
Ví dụ tag_to_ix: {'B-LOC': 0, 'B-MISC': 1, 'B-ORG': 2, 'B-PER': 3, 'I-LOC': 4, 'I-MISC': 5, 'I-ORG': 6, 'I-PER': 7, 'O': 8}


## Task 2: Tạo PyTorch Dataset và DataLoader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

### 1. Tạo lớp `NERDataset`

In [ ]:
class NERDataset(Dataset):
    def __init__(self, sentences, tags, word_to_ix, tag_to_ix):
        self.sentences = sentences
        self.tags = tags
        self.word_to_ix = word_to_ix
        self.tag_to_ix = tag_to_ix

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sentence = self.sentences[idx]
        tags = self.tags[idx]

        # Chuyển token -> index
        sentence_idx = [
            self.word_to_ix.get(word, self.word_to_ix["<UNK>"]) for word in sentence
        ]
        # Chuyển nhãn -> index
        tag_idx = [self.tag_to_ix[tag] for tag in tags]

        return torch.tensor(sentence_idx, dtype=torch.long), torch.tensor(tag_idx, dtype=torch.long)


### 2. Tạo `DataLoader`

In [ ]:
def collate_fn(batch):
    """
    batch: list các tuple (sentence_tensor, tag_tensor)
    """
    sentences, tags = zip(*batch)

    # Đệm (pad) câu và nhãn
    sentences_padded = pad_sequence(sentences, batch_first=True, padding_value=word_to_ix["<PAD>"])
    tags_padded = pad_sequence(tags, batch_first=True, padding_value=-1).long()

    return sentences_padded, tags_padded


In [ ]:
# Tạo Dataset
train_dataset = NERDataset(train_sentences, train_tags_str, word_to_ix, tag_to_ix)
val_dataset = NERDataset(val_sentences, val_tags_str, word_to_ix, tag_to_ix)
test_dataset = NERDataset(test_sentences, test_tags_str, word_to_ix, tag_to_ix)

# Tạo DataLoader
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=4, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=4, collate_fn=collate_fn)

for batch_sentences, batch_tags in train_loader:
    print("Batch sentences shape:", batch_sentences.shape)
    print("Batch tags shape:", batch_tags.shape)
    print("Ví dụ batch đầu tiên:")
    print("Sentences:\n", batch_sentences)
    print("Tags:\n", batch_tags)
    break


Batch sentences shape: torch.Size([4, 20])
Batch tags shape: torch.Size([4, 20])
Ví dụ batch đầu tiên:
Sentences:
 tensor([[10295,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [ 1728,  1584,  8945,  8941,    18,    41,  6347,    57,  8943,   121,
           163,  8944,    89,   237,  8946,   237,  5134,   163,  3195,    10],
        [18394,   712,  3149, 15722, 15271, 15386, 14974, 15714,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [ 8992,  1601, 18600, 18617,   511,    23,   258, 23056, 10039,    72,
           203,  1285, 23057,   500, 14433,   512,  5474,    10,     0,     0]])
Tags:
 tensor([[ 8, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
         -1, -1],
        [ 8,  8,  3,  7,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,
          0,  8],
        [ 2,  8,  8,  8,  8,  8,  8,  8, -1

## Task 3: Xây dựng Mô hình RNN

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class RNNForNER(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_size):
        super(RNNForNER, self).__init__()

        # 1. Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        # 2. LSTM layer
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        # 3. Fully connected layer
        self.fc = nn.Linear(hidden_dim * 2, output_size)

    def forward(self, x):
        # x: [batch_size, seq_len]
        embeds = self.embedding(x)                     # [batch_size, seq_len, embedding_dim]
        lstm_out, _ = self.lstm(embeds)                # [batch_size, seq_len, hidden_dim*2]
        logits = self.fc(lstm_out)                     # [batch_size, seq_len, output_size]
        return logits


In [ ]:
vocab_size = len(word_to_ix)
embedding_dim = 100
hidden_dim = 128
output_size = len(tag_to_ix)

## Task 4: Huấn luyện Mô hình

In [ ]:
model = RNNForNER(vocab_size, embedding_dim, hidden_dim, output_size)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss(ignore_index=-1)

In [ ]:
num_epochs = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for sentences, tags in train_loader:
        sentences, tags = sentences.to(device), tags.to(device)

        optimizer.zero_grad()

        outputs = model(sentences)  # [batch_size, seq_len, num_labels]

        # reshape: gộp batch và seq_len để phù hợp với yêu cầu CrossEntropyLoss
        loss = loss_fn(outputs.view(-1, output_size), tags.view(-1))


        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {avg_loss:.4f}")


Epoch [1/5] - Loss: 0.3822
Epoch [2/5] - Loss: 0.1296
Epoch [3/5] - Loss: 0.0495
Epoch [4/5] - Loss: 0.0174
Epoch [5/5] - Loss: 0.0075


## Task 5: Đánh giá Mô hình

In [ ]:
def evaluate(model, data_loader, device):
    model.eval()
    total_correct, total_tokens = 0, 0

    with torch.no_grad():
        for sentences, tags in data_loader:
            sentences, tags = sentences.to(device), tags.to(device)

            outputs = model(sentences)
            preds = torch.argmax(outputs, dim=-1)

            # Mask bỏ padding
            mask = tags != -1

            correct = (preds == tags) & mask
            total_correct += correct.sum().item()
            total_tokens += mask.sum().item()

    accuracy = total_correct / total_tokens
    return accuracy


In [ ]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=71fea0256224b1bb65903302a30c29c4ad8d2e08a7d4bb79470ef78b85042f58
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
from seqeval.metrics import classification_report

def evaluate_seqeval(model, data_loader, device, ix_to_tag):
    model.eval()
    true_tags, pred_tags = [], []

    with torch.no_grad():
        for sentences, tags in data_loader:
            sentences, tags = sentences.to(device), tags.to(device)
            outputs = model(sentences)
            preds = torch.argmax(outputs, dim=-1)

            for i in range(len(tags)):
                valid_indices = tags[i] != -1  # bỏ padding
                true_seq = [ix_to_tag[idx.item()] for idx in tags[i][valid_indices]]
                pred_seq = [ix_to_tag[idx.item()] for idx in preds[i][valid_indices]]
                true_tags.append(true_seq)
                pred_tags.append(pred_seq)

    print(classification_report(true_tags, pred_tags, digits=4))


In [ ]:
ix_to_tag = {v: k for k, v in tag_to_ix.items()}

val_acc = evaluate(model, val_loader, device)
print(f"Validation Accuracy: {val_acc:.4f}")

# Chi tiết từng chỉ số
evaluate_seqeval(model, val_loader, device, ix_to_tag)


Validation Accuracy: 0.9565
              precision    recall  f1-score   support

         LOC     0.8363    0.8508    0.8435      1837
        MISC     0.7807    0.7527    0.7664       922
         ORG     0.7612    0.6726    0.7142      1341
         PER     0.6944    0.7980    0.7426      1842

   micro avg     0.7639    0.7790    0.7714      5942
   macro avg     0.7681    0.7686    0.7667      5942
weighted avg     0.7667    0.7790    0.7711      5942



In [ ]:
test_acc = evaluate(model, test_loader, device)
print(f"Test Accuracy: {test_acc:.4f}")

# Chi tiết từng chỉ số
evaluate_seqeval(model, test_loader, device, ix_to_tag)

Test Accuracy: 0.9352
              precision    recall  f1-score   support

         LOC     0.7898    0.7794    0.7846      1668
        MISC     0.6277    0.6197    0.6237       702
         ORG     0.7022    0.5792    0.6348      1661
         PER     0.5814    0.7285    0.6467      1617

   micro avg     0.6757    0.6861    0.6808      5648
   macro avg     0.6753    0.6767    0.6724      5648
weighted avg     0.6842    0.6861    0.6810      5648



In [ ]:
def predict_sentence(model, sentence, word_to_ix, ix_to_tag, device):
    model.eval()
    words = sentence.split()

    # Chuyển từ sang chỉ số
    sentence_idx = [word_to_ix.get(w, word_to_ix["<UNK>"]) for w in words]
    input_tensor = torch.tensor([sentence_idx], dtype=torch.long).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        preds = torch.argmax(outputs, dim=-1).squeeze(0).cpu().numpy()

    predicted_tags = [ix_to_tag[p] for p in preds]

    print("\nDự đoán:")
    for w, t in zip(words, predicted_tags):
        print(f"{w:15s} → {t}")


In [ ]:
predict_sentence(model, "VNU University is located in Hanoi", word_to_ix, ix_to_tag, device)


Dự đoán:
VNU             → B-ORG
University      → I-ORG
is              → O
located         → O
in              → O
Hanoi           → B-LOC
